# Швидка перевірка впливу видимості на авторський AnyVisLoc

**Перший етап: 8 кадрів × 5 умов = 40 спроб.** По два кадри з кожної сцени,
вибрані детерміновано з попередньої вибірки 24 кадрів без урахування успішності.
`RUN_FULL=True` продовжує той самий експеримент до 24 × 5 = 120 спроб.

Авторські CAMP → RoMa → P3P/RANSAC, геометрія ділянок і поворот за yaw з
метаданих, вибір позиції за максимумом inliers. **Наші пороги прийняття
не застосовуються.** Це контроль впливу видимості за відомої геометрії,
а не перевірка GPS-denied навігації з шумним INS/VIO.

Змінюються тільки пікселі кадру. Пошук і ранжування CAMP повторюються для
кожної умови. Незмінні ознаки карти кешуються в RAM лише для одного кадру.
Справжня ділянка карти не підставляється замість пошуку.

Сильні умови повторюють попередні налаштування; екстремальні — стрес-тест,
а не фізично відкалібрований туман або дим. Висновок початкового етапу
попередній: вісім різних кадрів, не сорок незалежних спостережень.

Оновіть всю папку `scripts/visibility_localization` у Drive, перезапустіть
Python-сеанс, виберіть GPU та виконайте notebook зверху вниз.


## 1. Каталоги та параметри


In [ ]:
from pathlib import Path
WORK_ROOT = Path('/content/visibility_localization')
UPSTREAM_ROOT = WORK_ROOT/'Benchmark'
ARCHIVE_DIR = WORK_ROOT/'archives'
DATASET_ROOT = WORK_ROOT/'data'
CODE_ROOT = Path('/content/drive/MyDrive/PhD/proj/scripts')
PACKAGE_DIR = CODE_ROOT/'visibility_localization'
SAVE_RESULTS_TO_DRIVE = True
RESULTS_ROOT = Path('/content/drive/MyDrive/PhD/visibility_results')
BASELINE_ROOT = RESULTS_ROOT/'visibility-v5-aerial-roma-24'
RUN_TAG = 'author-visibility-quick-v1'
RUN_FULL = False  # True: продовжити з 8 до 24 кадрів без повторення готових спроб.
CAMP_LOCAL_PATH = None
SCENES = ['Scene_05','Scene_08','Scene_11','Scene_21']
ARCHIVES = {scene:f'/content/drive/MyDrive/PhD/proj/AnyVisLoc/{scene}.zip' for scene in SCENES}
SCENE_URLS = {}
SOURCES = {'dataset_portal':'https://1drv.ms/f/c/fef40436afe2e5bb/IgC0Zat90dsgTY-2lNlDliGXAfuTZOYEeKcllr7VPD_F3eY?e=LJ5fAD'}
CONFIG = {'seed': 42,
 'reference_mode': 'aerial',
 'matching_method': 'Roma',
 'map_alignment': 'none',
 'scenes': [],
 'nadir_max_deg': 15.0,
 'preferred_nadir_deg': 10.0,
 'sample_count': 24,
 'pose_convention': 'ned_attitude',
 'sigma_xy_m': [20.0],
 'trials': 1,
 'profiles': {'small': {'yaw_deg': 2.0, 'tilt_deg': 1.0, 'z_m': 2.0},
              'main': {'yaw_deg': 5.0, 'tilt_deg': 2.0, 'z_m': 5.0},
              'large': {'yaw_deg': 15.0, 'tilt_deg': 5.0, 'z_m': 10.0}},
 'active_profiles': ['main'],
 'retrieval_modes': ['prior'],
 'top_k': 5,
 'search_sigma': 3.0,
 'gallery_overlap': 0.5,
 'patch_scale': 1.0,
 'retrieval_size': 384,
 'retrieval_batch': 8,
 'max_gallery_tiles': 12000,
 'max_patch_pixels': 4000000,
 'fallback_patch_sizes_m': [64.0, 128.0, 256.0],
 'max_image_side': 1600,
 'match_keypoints': 3000,
 'gallery_cache_entries': 2,
 'match_resize_ratio': 0.4,
 'pnp_reprojection_px': 8.0,
 'pnp_iterations': 2000,
 'pnp_confidence': 0.999,
 'min_matches': 12,
 'min_inliers': 8,
 'min_inlier_ratio': 0.25,
 'max_median_reprojection_px': 4.0,
 'min_positive_depth_fraction': 0.95,
 'success_threshold_m': 10.0,
 'recall_overlap_thresholds': [0.25, 0.5, 0.75],
 'fog': {'fog_weak': 0.8, 'fog_moderate': 0.55, 'fog_strong': 0.3},
 'airlight_rgb': [0.92, 0.94, 0.96],
 'smoke': {'smoke_weak': {'coverage': 0.35,
                          'transmission': 0.55,
                          'softness': 0.08,
                          'color_rgb': [0.65, 0.63, 0.6]},
           'smoke_strong': {'coverage': 0.7,
                            'transmission': 0.2,
                            'softness': 0.08,
                            'color_rgb': [0.5, 0.48, 0.45]}}}
CONFIG['scenes'] = SCENES
print('Авторський контроль: aerial + CAMP + RoMa + P3P; yaw/геометрія з метаданих.')
print('Наші пороги якості не застосовуються; без синтетичного шуму prior.')
print('Етап:', '24 кадри / 120 спроб' if RUN_FULL else '8 кадрів / 40 спроб')


## 2. Drive, залежності й перевірки


In [ ]:
import os, sys, json, subprocess
from google.colab import drive

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
print(sys.version_info[:2])
drive.mount('/content/drive')

required_files = (
    'quick_visibility.py', 'test_quick_visibility.py', 'ablation.py', 'alignment.py', 'test_alignment.py', '__init__.py', 'core.py', 'data.py', 'backend.py', 'campaign.py',
    'colab.py', 'analysis.py', 'check_upstream.py', 'test_experiment.py',
    'requirements-colab.txt', 'requirements-roma-colab.txt', 'models.py', 'test_matcher_selection.py', 'test_batch_sampling.py',
)
missing = [name for name in required_files if not (PACKAGE_DIR/name).is_file()]
if missing:
    raise FileNotFoundError(
        f'У {PACKAGE_DIR} відсутні файли: {missing}. '
        'Скопіюйте всю локальну папку scripts/visibility_localization або виправте CODE_ROOT.'
    )

# Байткод і кеші не записуються до завантаженої папки в Drive.
sys.pycache_prefix = str(WORK_ROOT/'pycache')
os.environ['PYTHONPYCACHEPREFIX'] = sys.pycache_prefix
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))

import visibility_localization
loaded_package = Path(visibility_localization.__file__).resolve().parent
if loaded_package != PACKAGE_DIR.resolve():
    raise RuntimeError(f'Імпортовано іншу копію: {loaded_package}. Перезапустіть Python-сеанс.')
print('Модулі імпортовано з:', loaded_package)


In [ ]:
import sys, subprocess, importlib.metadata
from pathlib import Path
from packaging.requirements import Requirement

python_version = sys.version_info[:2]
if not (3, 10) <= python_version <= (3, 13):
    raise RuntimeError(f'Підтримується Python 3.10–3.13; поточний: {sys.version}')
if python_version == (3, 13):
    torch_version, vision_version, cuda_build = '2.6.0', '0.21.0', 'cu126'
    numpy_version = '2.1.3'
else:
    torch_version, vision_version, cuda_build = '2.5.1', '0.20.1', 'cu121'
    numpy_version = '1.26.4'
print(f'Python {sys.version.split()[0]}: torch {torch_version}, '
      f'torchvision {vision_version}, {cuda_build}, NumPy {numpy_version}', flush=True)
requirements = PACKAGE_DIR / 'requirements-colab.txt'
# Перевіряємо також копію requirements у Drive до будь-яких змін середовища.
active_requirements = [Requirement(line) for line in requirements.read_text().splitlines()
                       if line.strip() and not line.lstrip().startswith('#')]
if not any(r.name == 'numpy' and (r.marker is None or r.marker.evaluate())
           and r.specifier.contains(numpy_version) for r in active_requirements):
    raise RuntimeError('Оновіть requirements-colab.txt у Drive з локальної папки '
                       'scripts/visibility_localization та повторіть комірку.')

# Зміна бібліотеки, вже імпортованої в ядро, потребує перезапуску.
modules = {'numpy': 'numpy', 'scipy': 'scipy', 'pandas': 'pandas',
           'matplotlib': 'matplotlib', 'Pillow': 'PIL', 'shapely': 'shapely',
           'scikit-image': 'skimage', 'torch': 'torch', 'torchvision': 'torchvision',
           'opencv-python-headless': 'cv2', 'timm': 'timm', 'kornia': 'kornia'}
def installed_version(name):
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None
loaded_before = {name: installed_version(name) for name, module in modules.items()
                 if module in sys.modules}
pins = [f'torch=={torch_version}', f'torchvision=={vision_version}']
WORK_ROOT.mkdir(parents=True, exist_ok=True)
constraints = WORK_ROOT / 'torch-constraints.txt'
constraints.write_text('\n'.join(pins) + '\n')
# Без -q: справжня причина помилки pip залишиться у виводі комірки.
subprocess.run([sys.executable, '-m', 'pip', 'install', *pins,
                '--index-url', f'https://download.pytorch.org/whl/{cuda_build}'], check=True)
install_requirements = (PACKAGE_DIR/'requirements-roma-colab.txt'
                        if CONFIG.get('matching_method', 'SP_LG') == 'Roma' else requirements)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(install_requirements),
                '-c', str(constraints)], check=True)
changed = [name for name, version in loaded_before.items()
           if installed_version(name) != version]
restart_message = ('Runtime → Restart session, потім виконайте notebook зверху вниз.')
if changed:
    raise RuntimeError(f'Оновлено вже імпортовані бібліотеки: {changed}. ' + restart_message)
import numpy as np, torch, torchvision, cv2
if (np.__version__ != numpy_version
        or torch.__version__.split('+')[0] != torch_version
        or torchvision.__version__.split('+')[0] != vision_version
        or cv2.__version__ != '4.10.0'):
    raise RuntimeError('Версії в ядрі не відповідають встановленим. ' + restart_message)
assert torch.cuda.is_available(), 'GPU CUDA недоступний; перевірте тип runtime'
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)
# Зберігаємо stdout/stderr pytest: деякі клієнти не показують вивід дочірнього процесу.
test_command = [sys.executable, '-m', 'pytest', '-q', '--tb=short',
                str(PACKAGE_DIR/'test_experiment.py'), str(PACKAGE_DIR/'test_matcher_selection.py'),
                str(PACKAGE_DIR/'test_batch_sampling.py'), str(PACKAGE_DIR/'test_alignment.py'), str(PACKAGE_DIR/'test_quick_visibility.py'),
                '-o', 'cache_dir=/content/visibility_pytest_cache']
test_process = subprocess.run(test_command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, encoding='utf-8', errors='replace', check=False)
test_log = WORK_ROOT/'setup-pytest.log'
test_log.write_text(test_process.stdout, encoding='utf-8')
print(test_process.stdout, flush=True)
print('Звіт тестів:', test_log, flush=True)
if SAVE_RESULTS_TO_DRIVE:
    drive_test_log = Path(RESULTS_ROOT)/'setup-pytest.log'
    drive_test_log.parent.mkdir(parents=True, exist_ok=True)
    drive_test_log.write_text(test_process.stdout, encoding='utf-8')
    print('Копія звіту в Drive:', drive_test_log, flush=True)
if test_process.returncode:
    raise RuntimeError(f'Автоматичні тести не пройшли (код {test_process.returncode}). '
                       f'Причина — у звіті вище та у {test_log}. Наступні розділи не запускати.')
from visibility_localization.colab import checkout_upstream
upstream = checkout_upstream(UPSTREAM_ROOT)
from visibility_localization.check_upstream import verify
print('Синтетична перевірка авторських інтерфейсів:', verify(upstream, quick_visibility=True))
output_root = Path(RESULTS_ROOT)/RUN_TAG
output_root.mkdir(parents=True, exist_ok=True)


## 3. Дані та ті самі 24 кадри


In [ ]:
from visibility_localization.colab import official_scene_links,acquire_dataset
from visibility_localization.data import select_samples,read_sample,load_reference
existing=sorted({p.parent.name for p in Path(DATASET_ROOT).rglob('L??_reference.json')})
print('Уже розпаковані сцени:',existing)
if not CONFIG['scenes']:
    try:
        available=official_scene_links(None)
        print('ZIP в офіційному порталі:')
        for scene,item in sorted(available.items()): print(scene,item['name'],item.get('size'),'bytes')
    except Exception as exc:
        print('Інвентар недоступний:',type(exc).__name__,str(exc))
        print('Офіційний портал:',SOURCES['dataset_portal'])
        print('Резерв: ARCHIVES або вже розпакований DATASET_ROOT.')
    raise ValueError('Оберіть SCENES у розділі 2. Жоден ZIP автоматично не завантажено.')
print('Завантажуємо лише:',CONFIG['scenes'])
print('ZIP ->',ARCHIVE_DIR,'; розпакування ->',DATASET_ROOT)
acquisition=acquire_dataset(CONFIG['scenes'],ARCHIVE_DIR,DATASET_ROOT,
    output_root/'acquisition.json',archives=ARCHIVES,urls=SCENE_URLS)
print(json.dumps(acquisition,ensure_ascii=False,indent=2))


In [ ]:
import pandas as pd
from visibility_localization.data import validate_balanced_selection
from visibility_localization.quick_visibility import pilot_order
selected, audit = select_samples(DATASET_ROOT, CONFIG, output_root/'selection')
display(validate_balanced_selection(selected, SCENES, 24))
baseline = pd.read_csv(BASELINE_ROOT/'selection/selected_samples.csv')
keys = ['scene_id','sample_id']
if set(map(tuple, baseline[keys].to_numpy())) != {(r['scene_id'],r['sample_id']) for r in selected}:
    raise RuntimeError('Вибірка не збігається з попередніми 24 кадрами')
ordered = pilot_order(selected, SCENES, CONFIG['seed'])
display(pd.DataFrame(ordered if RUN_FULL else ordered[:8])[keys])


## 4. Видимість: швидко перевіряємо сильні умови

| Умова | Параметри |
|---|---|
| clear | Оригінал |
| fog_strong | Однорідне пропускання 0,30 |
| fog_extreme | Однорідне пропускання 0,10 |
| smoke_strong | Охоплення 70%, мінімальне пропускання 0,20 |
| smoke_extreme | Охоплення 90%, мінімальне пропускання 0,05 |

Однакове випадкове поле диму для обох рівнів одного кадру. Маска й її
параметри відновлюються за seed; середнє пропускання записується в результат.


In [ ]:
import matplotlib.pyplot as plt
from visibility_localization.core import degrade, stable_seed, digest
from visibility_localization.quick_visibility import CONDITIONS, visibility_config
example = read_sample(ordered[0]['path'], max_side=1000000)
pair_id = digest([example['scene_id'],example['sample_id']])[:24]
smoke_seed = stable_seed(CONFIG['seed'],pair_id,'smoke')
fig, axes = plt.subplots(1,5,figsize=(18,4))
for ax,condition in zip(axes,CONDITIONS):
    image,_,_ = degrade(example['rgb'],condition,visibility_config(CONFIG),smoke_seed)
    ax.imshow(image); ax.set_title(condition); ax.axis('off')
plt.show(); plt.close(fig)
del example, image, fig


## 5. Офіційні моделі

Використовується наявна ініціалізація CAMP/RoMa й кеш ваг у Drive.
Backend тут надає моделі; його адаптований пошук і правила прийняття
не викликаються. Експеримент окремо використовує функції з `run_avl.py`.

ConvNeXt створюється без початкового мережевого завантаження: його повний
стан береться з уже наявного checkpoint CAMP. Повнота ключів і розміри
перевіряються до завантаження; за неповного checkpoint запуск зупиняється.
Хеш стану CAMP звіряється з попереднім експериментом перед продовженням.


In [ ]:
# CAMP checkpoint містить навчені ваги backbone. Перевіряємо повноту
# перед заміною зайвого завантаження початкових ImageNet-ваг.
from contextlib import contextmanager
import importlib

@contextmanager
def camp_from_complete_checkpoint(source, checkpoint):
    import torch
    if str(source) not in sys.path:
        sys.path.insert(0, str(source))
    module = importlib.import_module('Retrieval_Models.CAMP.hand_convnext.ConvNext.make_model')
    original_factory = module.create_model
    def from_checkpoint(name, *args, **kwargs):
        if name != 'convnext_base':
            raise RuntimeError(f'Неочікуваний backbone CAMP: {name}')
        kwargs['pretrained'] = False
        model = original_factory(name, *args, **kwargs)
        checkpoint_state = torch.load(str(checkpoint), map_location='cpu', weights_only=True)
        prefix = 'model_1.convnext.'
        backbone = {key[len(prefix):]:value for key,value in checkpoint_state.items()
                    if key.startswith(prefix)}
        expected = model.state_dict()
        missing = sorted(set(expected)-set(backbone))
        wrong_shape = [key for key in expected if key in backbone
                       and tuple(expected[key].shape) != tuple(backbone[key].shape)]
        unexpected = sorted(set(backbone)-set(expected))
        if missing or wrong_shape or unexpected:
            raise RuntimeError('CAMP checkpoint не покриває backbone повністю; '
                               f'missing={missing}, wrong_shape={wrong_shape}, unexpected={unexpected}. '
                               'Ініціалізацію з неповними вагами зупинено.')
        model.load_state_dict(backbone, strict=True)
        print('ConvNeXt: усі параметри отримано з повного CAMP checkpoint; '
              'завантаження ImageNet-ваг не потрібне.', flush=True)
        return model
    module.create_model = from_checkpoint
    try:
        yield
    finally:
        module.create_model = original_factory

# Постійний кеш RoMa/DINOv2 переживає завершення сеансу Colab.
import os
MODEL_CACHE = Path('/content/drive/MyDrive/PhD/model_cache/anyvisloc')
os.environ['VISIBILITY_MODEL_CACHE'] = str(MODEL_CACHE)
print('Постійний кеш ваг:', MODEL_CACHE)
from visibility_localization.colab import CAMP_URL
checkpoint = Path(CAMP_LOCAL_PATH) if CAMP_LOCAL_PATH else Path(UPSTREAM_ROOT)/'Retrieval_Models/CAMP/weights/weights_0.9446_for_U1652.pth'
if not checkpoint.is_file():
    import gdown
    checkpoint.parent.mkdir(parents=True, exist_ok=True)
    downloaded = gdown.download(CAMP_URL, str(checkpoint), fuzzy=True, quiet=False)
    if not downloaded:
        raise RuntimeError('CAMP не завантажено. Задайте CAMP_LOCAL_PATH із Google Drive.')
from visibility_localization.backend import AnyVisLocBackend
with camp_from_complete_checkpoint(upstream, checkpoint):
    backend = AnyVisLocBackend(upstream, CONFIG, checkpoint)
print(json.dumps(backend.identity, ensure_ascii=False, indent=2))

print('Фактичний matcher:', backend.identity['matching_method'], '; карта/DSM:', CONFIG['reference_mode'])
assert backend.identity['matching_method'] == CONFIG['matching_method']

# Дозволяємо продовження лише з точно таким самим станом CAMP.
previous_manifest = output_root/'experiment_config.json'
if previous_manifest.is_file():
    previous_identity = json.loads(previous_manifest.read_text(encoding='utf-8'))['backend']
    for identity_key in ['camp_checkpoint_sha256','camp_state_sha256']:
        if backend.identity[identity_key] != previous_identity[identity_key]:
            raise RuntimeError(f'Стан CAMP відрізняється: {identity_key}. '
                               'Продовження попередніх результатів зупинено.')
    print('Хеш CAMP збігається з попереднім запуском: можна продовжувати.')


## 6. Парний запуск

На кожному кадрі спочатку clear, потім погіршення. Перший кадр дає оцінку
часу за фактичним виконанням. Авторський глобальний пошук може бути
повільнішим за наш локальний; час заздалегідь не гарантується.

JSON записується після кожної спроби. Для продовження використовуйте
ті самі параметри й RUN_TAG; `RUN_FULL` можна змінити на True.
Технічна помилка зупиняє експеримент і зберігає traceback.

Моніторинг RAM/GPU записується кожні 2 секунди в `resource_log.jsonl`.
Якщо доступно, `cgroup_events.jsonl` зберігає лічильники OOM операційної системи.
Відкривши notebook безпосередньо в браузерному Colab, можна окремо
перевірити виконання без з’єднання через розширення VS Code.
RUN_TAG і модулі залишаються попередніми; готові записи використовуються повторно.


In [ ]:
# Моніторинг не змінює моделі, seed або ідентичність експерименту.
# Запис кожні 2 с, щоб побачити короткі піки RAM/GPU всередині кадру.
from threading import Event, Thread
from visibility_localization.campaign import log_resources
from visibility_localization.quick_visibility import run_campaign

def resource_monitor(stop, backend, output):
    while not stop.is_set():
        try:
            log_resources(output, 'periodic_monitor', backend)
            # Лічильник OOM у cgroup може відрізнити kill ОС від втрати з'єднання.
            import json, time
            events = Path('/sys/fs/cgroup/memory.events')
            if events.is_file():
                with (output/'cgroup_events.jsonl').open('a',encoding='utf-8') as stream:
                    stream.write(json.dumps({'time_unix':time.time(),
                        'events':events.read_text()})+'\n')
        except Exception as exc:
            # Збій телеметрії не повинен зупинити локалізацію.
            print('Помилка моніторингу ресурсів:', repr(exc), flush=True)
            return
        stop.wait(2.)

output_root.mkdir(parents=True,exist_ok=True)
monitor_stop = Event()
monitor_thread = Thread(target=resource_monitor,args=(monitor_stop,backend,output_root),daemon=True)
monitor_thread.start()
try:
    results = run_campaign(backend, UPSTREAM_ROOT, DATASET_ROOT, selected, CONFIG,
                           output_root, full=RUN_FULL)
finally:
    monitor_stop.set()
    monitor_thread.join(timeout=5.)
print('Збережено спроб:',len(results))
display(results[['scene_id','sample_id','condition','status','error_xy_m','matches','inliers','inlier_ratio']])


## 7. Чи є вплив?

Acc@3/5/10 рахується від усіх спроб, невдалий PnP не вилучається.
Похибка має значення лише там, де отримана позиція; окремо показані
втрати/відновлення точності відносно clear для того самого кадру.

Зменшення inliers або зростання похибки може проявитися раніше, ніж
втрата локалізації. Bootstrap-інтервал зміни inliers рахується за кадрами,
а не за всіма варіантами зображень разом; на 8 кадрах він попередній.
Зниження inliers саме по собі не означає втрату точності позиції.
Якщо результат неоднозначний, увімкніть RUN_FULL=True й повторіть розділи 1 і 6–7.


In [ ]:
from visibility_localization.quick_visibility import summarize
overview, paired = summarize(results,output_root)
display(overview)
print('Кожний кадр порівняно з його clear:')
display(paired[['scene_id','sample_id','condition','error_xy_m_clear','error_xy_m',
                'inliers_clear','inliers','inlier_ratio_clear','inlier_ratio']])
fig,axes = plt.subplots(1,3,figsize=(16,4))
for pair_id,group in results.groupby('pair_id'):
    group=group.set_index('condition').reindex(CONDITIONS)
    axes[0].plot(range(5),group['inliers'],alpha=.5,marker='.')
    axes[1].plot(range(5),group['error_xy_m'],alpha=.5,marker='.')
axes[0].set_ylabel('Inliers'); axes[1].set_ylabel('Похибка XY, м')
axes[2].plot(range(5),overview.set_index('condition').reindex(CONDITIONS).accuracy_10m*100,marker='o')
axes[2].set_ylabel('Acc@10, %'); axes[2].set_ylim(0,105)
for ax in axes:
    ax.set_xticks(range(5),CONDITIONS,rotation=35,ha='right'); ax.grid(alpha=.25)
fig.tight_layout(); fig.savefig(output_root/'visibility_quick.png',dpi=150)
plt.show(); plt.close(fig)
print('Результати:',output_root)
